In [44]:
import os
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [45]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
pilot2_dir = os.getcwd()

# data/processed 폴더 위치 지정
processed_data_dir = pilot2_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')


# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = pilot2_dir + ('\\graph' if os_system == 'Windows' else '/graph')

In [46]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
pilot_data = pd.read_csv(processed_data_dir + 'similarity_coherence_data_300_with_words.csv', index_col=0, keep_default_na=False)
pilot_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,similarity_friend30_friend,coherence_key_key,coherence_key_money,coherence_key_friend,coherence_money_key,coherence_money_money,coherence_money_friend,coherence_friend_key,coherence_friend_money,coherence_friend_friend
Prolific_ID,,,,,,,,,,,,,,,,,,,,,
5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,roofing,...,0.19990666,0.041143,0.102066,0.075201,0.030420,0.143791,0.063515,0.037806,0.096950,0.146998
5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time space,possibilites,infinite,time,goals,...,0.17192948,0.096018,0.140075,0.120077,0.053525,0.126333,0.125166,0.081238,0.108360,0.105842
5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,said,dressing,clothes,old,...,0.058080602,0.079032,0.117155,0.169555,0.057519,0.099424,0.132149,0.049724,0.145488,0.082121


In [52]:
def get_smoothed_similarity_func(similarity_df: pd.DataFrame, i_subject: int, topic: str):
    # trial 번호와 해당 trial에 대한 similarity 값을 배열로 변환
    trial_numbers = np.array(list(range(1, 31)))

    # 특정 피험자의 유사도 점수들 받아오기
    similarity_values = similarity_df.iloc[i_subject].tolist()
    similarity_values = np.array([float(value) if value != '' else 0.0 for value in similarity_values])

    # 데이터를 보간하는 함수 생성
    interpolation_function = interp1d(trial_numbers, similarity_values, kind='quadratic')

    # 정수값에 대한 데이터 추출
    integer_trial_numbers = np.arange(1, 31)
    integer_similarity_values = interpolation_function(integer_trial_numbers)

    # 부드러운 곡선을 위해 trial 번호를 더 자세히 나누기
    fine_trial_numbers = np.linspace(1, 30, 300)
    smoothed_similarity_values = interpolation_function(fine_trial_numbers)

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_similarity_curve.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_similarity_curve.png')

    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(fine_trial_numbers, smoothed_similarity_values, label='Interpolated Curve', color='r')
    plt.scatter(integer_trial_numbers, integer_similarity_values, label='Integer Data', marker='o', color='b')
    plt.xlabel('Trial')
    plt.ylabel('Similarity Value')
    plt.title(f'Subject {i_subject}: Smoothed Similarity Curv')
    plt.legend()
    plt.grid(True)
    plt.savefig(graph_image_path)
    # plt.show()
    
    # 그래프 표시하지 않음
    plt.close()

    return integer_trial_numbers, integer_similarity_values


In [53]:
def calculate_slope(x, y):
    """
    x 값이 정수인 경우에만 데이터 포인트 간의 기울기(변화율)를 계산합니다.

    매개변수:
    x (numpy 배열 또는 리스트): x 값 배열
    y (numpy 배열 또는 리스트): y 값 배열

    반환값:
    slopes (numpy 배열): 데이터 포인트 간의 기울기 배열
    """
    n = len(x)
    slopes = np.zeros(n)

    for i in range(1, n - 1):
        if np.floor(x[i]) == x[i]: # x가 정수일떄만 기울기 계산(실제 데이터포인트)
            slopes[i] = (y[i + 1] - y[i - 1]) / (x[i + 1] - x[i - 1])

    # 첫 번째와 마지막 데이터 포인트의 기울기는 계산할 수 없으므로 0으로 설정
    slopes[0] = 0.0
    slopes[n - 1] = 0.0

    return slopes


In [54]:
def get_trend_graph(i_subject: int, topic: str, trial_numbers: list, similarity_values: list):

    # 데이터 포인트 간의 기울기 계산
    trial = np.array(trial_numbers)
    y = np.array(similarity_values)
    slopes = calculate_slope(trial, y)

    slope = np.array(slopes)

    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(trial, slope, label='Slope vs. Trial', color='b')
    plt.axhline(0, color='r', linestyle='--', label='Zero Line') 

    # 각 데이터 포인트의 값 표시
    for i, s in enumerate(slope):
        plt.text(trial[i], s, f'{s:.3f}', ha='center', va='bottom', fontsize=8, color='black')

    # 추세선 (회귀선) 그리기
    z = np.polyfit(trial, slope, 1)
    p = np.poly1d(z)
    plt.plot(trial, p(trial), color='g', linestyle='-', label='Trend Line')

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_trend_graph.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_trend_graph.png')

    # 그래프 표시
    plt.xlabel('Trial')
    plt.ylabel('Slope')
    plt.title(f'Subject {i_subject}: Trend of similarity according to trial')
    plt.legend()
    plt.grid(True)
    plt.ylim(-0.15, 0.15)
    plt.savefig(graph_image_path)
    # plt.show()

    # 그래프 표시하지 않음
    plt.close()

In [56]:
similarity_key_key = pilot_data.iloc[:, 91:121]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_key_key,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'key_key')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'key_key',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [57]:
similarity_key_money = pilot_data.iloc[:, 121:151]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_key_money,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'key_money')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'key_money',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [59]:
similarity_key_friend = pilot_data.iloc[:, 151:181]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_key_friend,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'key_friend')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'key_friend',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [61]:
similarity_money_key = pilot_data.iloc[:, 181:211]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_money_key,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'money_key')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'money_key',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [63]:
similarity_money_money = pilot_data.iloc[:, 211:241]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_money_money,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'money_money')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'money_money',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [65]:
similarity_money_friend = pilot_data.iloc[:, 241:271]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_money_friend,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'money_friend')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'money_friend',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [67]:
similarity_friend_key = pilot_data.iloc[:, 271:301]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_friend_key,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'friend_key')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'friend_key',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [69]:
similarity_friend_money = pilot_data.iloc[:, 301:331]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_friend_money,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'friend_money')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'friend_money',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)

In [71]:
similarity_friend_friend = pilot_data.iloc[:, 331:361]

for i_subject in range(0, 58):
    integer_trial_numbers, integer_similarity_values = get_smoothed_similarity_func(similarity_df = similarity_friend_friend,
                                                                                    i_subject = i_subject,
                                                                                    topic = 'friend_friend')
    get_trend_graph(i_subject = i_subject, 
                    topic = 'friend_friend',
                    trial_numbers = integer_trial_numbers,
                    similarity_values = integer_similarity_values)